# L13c: The Model Context Protocol
In this lecture, we introduce the Model Context Protocol (MCP), an open standard that connects language model applications to tools. We define the roles in an MCP system, follow the messages of a tool call, and look at what the host and the server check before a tool runs.

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
>
> * __Describe the roles in an MCP system:__ A host connects a large language model (LLM) to servers, which offer tools. We'll define the host, the client, and the server.
> * __Read the messages of a tool call:__ A client and a server exchange [JSON-RPC](https://www.jsonrpc.org/specification) messages. We'll follow the messages that discover a server, list its tools, and call a tool.
> * __Identify the trust boundaries of a tool call:__ A tool runs code on the user's computer, and the LLM writes the arguments of each call. We'll look at what the host and the server each check.

Let's get started!

___

## Examples
We will use the following example to illustrate the key ideas of this lecture:

> [▶ Build an MCP server for the National Weather Service API](CHEME-5800-L13c-Example-MCP-NWS-Fall-2026.ipynb). In this example, we wrap our software development kit (SDK) for the National Weather Service (NWS) API in an MCP server with two tools. We call each tool as a host would, and send two requests that fail in different ways.

___

## Language Models and Tools
A __large language model (LLM)__ is a neural network that reads and writes text. On its own, an LLM cannot look up today's weather or run our Julia code. An application gives an LLM access to __tools__, functions that the application runs when the LLM asks for them.

In a __tool call__, the LLM replies with the name of a tool and its arguments instead of an answer. The application runs the tool and gives the result to the LLM. The LLM then writes its answer from the result.

The __Model Context Protocol (MCP)__ is an open standard that connects applications to tools.

Without a standard, each application needs its own connection code for each program that offers tools. For $M$ applications and $N$ programs, that is up to $M\times{N}$ pieces of code. With MCP, each application and each program implements the protocol once, for $M+N$ implementations.

Anthropic released MCP in November 2024 and donated it to the Agentic AI Foundation of the Linux Foundation in December 2025. We use [version `2026-07-28` of the MCP specification](https://modelcontextprotocol.io/specification/2026-07-28).

___

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  body[data-jp-theme-light="true"] .course-diagram { color-scheme: light; }
  body[data-jp-theme-light="false"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>

## Hosts, Clients, and Servers
An MCP system has three roles:

<p align="center">
    <img class="course-diagram" src="figs/Fig-MCP-Architecture/Fig-MCP-Architecture.svg" width="1100" style="max-width:100%; height:auto;" alt="A User box on the left is joined by a two-way arrow to a large blue box labeled Host. Inside the host, a Language model box is joined by gray lines to two Client boxes. A blue two-way arrow labeled MCP messages joins the upper client to a Weather server box, and another joins the lower client to a FRED server box. A dark two-way arrow labeled HTTPS joins the weather server to an NWS API box, and another joins the FRED server to a FRED API box. Notes below say that the host is a chat app, a coding assistant, or our notebook; the servers are our Julia code, started by the host; and the APIs are RESTful web services."/>
</p>

* __Host:__ The host is the application that the user works with, such as a chat app or a coding assistant. The host sends the user's questions to the LLM, starts the servers, and decides which tool calls to run.
* __Client:__ Each client runs inside the host and exchanges MCP messages with one server.
* __Server:__ A server offers tools. The tools of our Julia servers call the NWS API and the Federal Reserve Economic Data (FRED) API over HTTPS.

A __transport__ carries messages between a client and a server. Our servers use the __stdio__ (standard input and output) transport: the host starts the server as a subprocess, writes requests to the server's __standard input__, and reads responses from its __standard output__.
Our servers write their log messages to __standard error__, a separate output stream.

For a server on another computer, the protocol also defines the __Streamable HTTP__ transport.

___

## MCP Messages
Earlier this week, we sent HTTP requests to the NWS API. Each request named an HTTP method and a URL, and each response carried a status code and a JSON body. Like the NWS responses, MCP messages are JSON text, but they follow [__JSON-RPC 2.0__](https://www.jsonrpc.org/specification), a standard format for requests and responses.

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  body[data-jp-theme-light="true"] .course-diagram { color-scheme: light; }
  body[data-jp-theme-light="false"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>

<p align="center">
    <img class="course-diagram" src="figs/Fig-MCP-ToolCall/Fig-MCP-ToolCall.svg" width="1000" style="max-width:100%; height:auto;" alt="A sequence diagram with four dashed lifelines: LLM, Host, MCP server, and Web API. Four shaded bands group twelve numbered arrows. Band one, Discover and list: step 1, a blue arrow from the host to the MCP server, server/discover request; step 2, a blue arrow back, server/discover result; step 3, a blue arrow from the host to the server, tools/list request; step 4, a blue arrow back, tools/list result: the tool list. A note on the Host lifeline then reads: User asks: Will it rain in Ithaca tonight? Band two, Choose a tool: step 5, a gray arrow from the host to the LLM, question and tool list; step 6, a gray arrow from the LLM to the host, tool name and arguments. Band three, Run the tool: step 7, a blue arrow from the host to the server, tools/call request; step 8, a dark arrow from the server to the web API, two HTTPS GET requests; step 9, a dark arrow back to the server, JSON forecast; step 10, a blue arrow from the server to the host, tools/call result. Band four, Write the answer: step 11, a gray arrow from the host to the LLM, tool result; step 12, a gray arrow from the LLM to the host, answer. A note at the bottom reads: Host shows the answer to the user. A legend names the arrow colors: gray, inside the host; blue, MCP message; dark, HTTPS request or response."/>
</p>

__One tool call.__ When the host starts the server, the host sends the `server/discover` and `tools/list` requests and gets the tool list (1–4). When the user asks a question, the LLM chooses a tool from that list (5–6), the server runs the tool (7–10), and the LLM writes its answer from the result (11–12).

We'll look at the message format, and then follow the three requests in order: `server/discover`, `tools/list`, and `tools/call`. The [messages folder](messages) holds each request and response that we saved from our weather server.

### The Message Format
A REST request names its action with an HTTP method and a URL, while a [JSON-RPC](https://www.jsonrpc.org/specification) request names its action in the `method` field.

Every MCP message is a [JSON-RPC 2.0](https://www.jsonrpc.org/specification) object. A __request__ carries `jsonrpc` (always `"2.0"`), an `id`, the `method` to run, and its `params` (parameters). The __response__ repeats the `id` and holds either a `result` or an `error`.

[Version `2026-07-28`](https://modelcontextprotocol.io/specification/2026-07-28) is __stateless__: every request names the protocol version, the client, and the features that the client supports. The server checks the version on every request and does not need to remember earlier requests.

### Discover, List, and Call
A host sends a server three kinds of requests:

| Request | What the host asks | What the response holds | Saved messages |
| --- | --- | --- | --- |
| `server/discover` | Which protocol versions does the server support, and what does it offer? | The supported versions, the features that the server offers, called __capabilities__ (ours offers tools), `instructions` for the LLM, and the server's name | [Request](messages/1-discover-request.json) · [Response](messages/1-discover-response.json) |
| `tools/list` | Which tools does the server have? | Each tool's `name`, `description`, and `inputSchema` | [Request](messages/2-list-request.json) · [Response](messages/2-list-response.json) |
| `tools/call` | Run this tool with these arguments, e.g., a one-hour forecast for Ithaca, NY | The result as text in `content` for the LLM, the same result as JSON data in `structuredContent` for a program, and `isError` set to `false` | [Request](messages/3-call-request.json) · [Response](messages/3-call-response.json) |

The LLM reads the tool descriptions to choose a tool. Each input schema is written in __JSON Schema__, a standard format that lists the arguments, their types, and which arguments are required. For example, `get_hourly_forecast` requires the numbers `latitude` and `longitude`, and accepts an optional integer `hours` from `1` to `48`.

### Two Kinds of Errors
A request can fail in two ways:

| Error | What happened | What the response holds | Saved messages |
| --- | --- | --- | --- |
| __Protocol error__ | The server cannot process the request, e.g., the request names a `get_snowfall` tool, which the server does not have | An `error` with a numeric `code` (here `-32602`) and a `message`, and no `result` | [Request](messages/4-unknown-tool-request.json) · [Response](messages/4-unknown-tool-response.json) |
| __Tool execution error__ | The tool ran but failed, e.g., the NWS API has no forecast for Paris, France, because it covers only the United States | A `result` with the error text in `content` and `isError` set to `true` | [Request](messages/5-paris-request.json) · [Response](messages/5-paris-response.json) |

A protocol error usually means a bug in the host or the server. A tool execution error gives the error text to the LLM, which can try different arguments or explain the failure to the user.

### The Server Loop
Our servers run a loop. Each pass of the loop reads one message and writes at most one response:

__Initialize__: Given the protocol version $v=$ [`2026-07-28`](https://modelcontextprotocol.io/specification/2026-07-28) and a set of tools $\mathcal{T}$, each with a name, a description, an input schema, and a handler (the function that runs the tool).

While the standard input is open __do__:
1. __Read:__ Get the next line $\ell$ of the standard input that is not blank.
2. __Check:__ Parse the line, $m\gets\texttt{parse}(\ell)$, and set the response $r$ by the first of these conditions that holds:
    - If $\ell$ is not valid JSON, __then__ $r\gets$ a parse error, `-32700`.
    - If $m$ is not a [JSON-RPC 2.0](https://www.jsonrpc.org/specification) message, __then__ $r\gets$ an invalid-request error, `-32600`.
    - If $m$ has no `id`, __then__ $r\gets\texttt{nothing}$, because a notification receives no response.
    - If $m$ does not name the protocol version $v$, __then__ $r\gets$ an unsupported-version error, `-32022`.
3. __Route:__ If none of the conditions in step 2 holds, __then__ choose $r$ by the method of $m$:
    - If the method is `server/discover`, __then__ $r\gets$ the protocol versions, capabilities, and name of the server.
    - If the method is `tools/list`, __then__ $r\gets$ the name, description, and input schema of each tool in $\mathcal{T}$.
    - If the method is `tools/call` and $\mathcal{T}$ has no tool with the requested name, __then__ $r\gets$ an invalid-params error, `-32602`.
    - If the method is `tools/call` and $\mathcal{T}$ has the requested tool, __then__ run its handler and set $r\gets$ its result. If the handler fails, __then__ $r$ holds the error text, with `isError` set to `true` (tool execution error).
    - Otherwise, $r\gets$ a method-not-found error, `-32601`.
4. __Write:__ If $r\neq\texttt{nothing}$, __then__ write $r$ as one line of JSON text to the standard output.

__Output__: The responses on the standard output. The loop ends when the client closes the standard input of the server.

[The `process_message(...)` function](src/Server.jl) carries out steps 2 and 3 for one line, and [the `serve(...)` function](src/Server.jl) runs the loop.

___

## Trust and Security
A __trust boundary__ is a point where data passes between parts of a system that we trust to different degrees. A tool call in our weather example crosses three trust boundaries:

* __LLM → host:__ The LLM asks for a tool and gives its arguments. The host decides whether to run the call, and asks the user for approval when its settings require it.
* __Host → server:__ Our stdio servers run on the user's computer with the user's permissions. A server can read or change any file that the user can. Install only servers whose code you trust.
* __Server → API:__ The server checks the tool arguments before it makes an API request. The server never puts an API key in an MCP message.

__Prompt injection__ happens when a tool result contains text that tells the LLM to do something the user did not ask for.

Our tools only read public data. Before each call to a tool that changes data, such as a tool that places an order, the host should ask the user for approval.

___

## Summary
In this lecture, we introduced the Model Context Protocol, the messages of a tool call, and the trust boundaries of a tool call.

> __Key Takeaways:__
>
> * __Hosts, clients, and servers:__ A host connects an LLM to MCP servers, with one client for each server. The LLM never talks to a server directly, because every tool call passes through the host.
> * __Discover, list, and call:__ A client discovers a server, lists its tools, and calls a tool with [JSON-RPC](https://www.jsonrpc.org/specification) requests that name the protocol version. A request that the server cannot process gets a protocol error, while a tool that fails returns a result with `isError` set to `true`.
> * __Trust boundaries:__ A server runs code with the permissions of the user, and the LLM writes the arguments of each call. The host decides whether to run each call, and the server checks every argument.

Any MCP host that supports [version `2026-07-28`](https://modelcontextprotocol.io/specification/2026-07-28) can now call our Julia code as tools.

___